In [ ]:
# ==============================================================================
# Celda 1: Preparación del Entorno y Módulos
# ==============================================================================
import pandas as pd
import yaml
import os
import sys
import json

# Configuración de rutas para acceder a la carpeta src
sys.path.append(os.path.abspath('../src'))

# Importamos la clase DataManager desde nuestro archivo script en src
from C05_data_manager import DataManager

# Configuración global de visualización
pd.set_option('display.max_columns', None)

print("✅ Entorno listo. Módulo DataManager cargado desde la carpeta src.")

In [ ]:
# ==============================================================================
# Celda 2: Inicialización del DataManager y Carga de Datos Interim
# ==============================================================================
# Definimos la ruta al config relativa al notebook
config_path = '../config/config.yaml'

# Instanciar el gestor indicando que la base de los datos está un nivel arriba (../)
dm = DataManager(config_path=config_path, base_path="../")

# Ahora la carga debería funcionar sin error de FileNotFoundError
df_ventas, df_precios = dm.load_interim_data()

print(f"📊 Datos cargados correctamente:")
print(f"   - Ventas: {df_ventas.shape[0]} registros")
print(f"   - Precios: {df_precios.shape[0]} registros")

In [ ]:
# ==============================================================================
# Celda 3: Fusión de Datasets (Merge Diario)
# ==============================================================================
# Unificar la información para tener Unidades, Precio y Costo en una sola tabla
# El manager utiliza las llaves definidas en el config.yaml
df_daily = dm.merge_data(df_ventas, df_precios)

print(f"🔗 Unión completada. Dimensiones del set diario: {df_daily.shape}")
display(df_daily.head())

In [ ]:
# ==============================================================================
# Celda 4: Validación de Integridad y Detección de Nulos
# ==============================================================================
# Verificamos si quedaron registros sin precio después de la unión
null_check = df_daily.isnull().sum()

if null_check.any():
    print("⚠️ ADVERTENCIA: Se detectaron valores nulos tras la unión. Revisa la imputación:")
    print(null_check[null_check > 0])
else:
    print("✅ Integridad confirmada: No existen registros huérfanos tras la unión.")

In [ ]:
# ==============================================================================
# Celda 5: Generación de Log de Auditoría de la Unión (JSON)
# ==============================================================================
# Guardamos el estado de la unión indicando explícitamente el nombre del dataframe
dm.save_merge_log(df_ventas, df_precios, df_daily, df_name="df_daily")

# Verificación de que el nombre del dataframe quedó registrado
log_path = os.path.join(dm.base_path, "outputs", "reports", "c05_status_transformation.json")
with open(log_path, 'r', encoding='utf-8') as f:
    check_log = json.load(f)

print(f"\n✅ Auditoría confirmada para el objeto: {check_log['dataframe_name']}")
print(f"   Timestamp del proceso: {check_log['timestamp']}")

In [ ]:
# ==============================================================================
# Celda 6: Preparación del Módulo de Transformación (C10)
# ==============================================================================
# Importamos el transformador desde la carpeta src
from C10_transformation import TimeSeriesTransformer

# Instanciamos el objeto con la configuración actual
transformer = TimeSeriesTransformer(dm.config)

print("✅ Módulo C10_transformation cargado y configurado exitosamente.")

In [ ]:
# ==============================================================================
# Celda 7: Transformación de Granularidad (Diario -> Mensual)
# ==============================================================================
# Aplicamos la agregación ponderada y el conteo de días con actividad
df_monthly = transformer.aggregate_to_monthly(df_daily)

print(f"📈 Transformación finalizada:")
print(f"   - Registros diarios procesados: {df_daily.shape[0]}")
print(f"   - Registros mensuales generados: {df_monthly.shape[0]}")
print(f"   - Reducción de dimensionalidad: {round((1 - len(df_monthly)/len(df_daily))*100, 2)}%")

display(df_monthly.head())

In [ ]:
# ==============================================================================
# Celda 8: Validación de Consistencia (Check de Sumas)
# ==============================================================================
# Regla de Oro: La suma total de unidades vendidas no debe cambiar tras la agregación
total_daily = df_daily[dm.config['column_mapping']['target_col']].sum()
total_monthly = df_monthly[dm.config['column_mapping']['target_col']].sum()

diff = abs(total_daily - total_monthly)

print(f"🔍 Auditoría de Sumas:")
print(f"   - Suma Unidades (Diario):  {total_daily:,.0f}")
print(f"   - Suma Unidades (Mensual): {total_monthly:,.0f}")

if diff < 1e-5:
    print("✅ EXCELENTE: No hay pérdida de unidades en la transformación.")
else:
    print(f"❌ ALERTA: Diferencia detectada de {diff} unidades. Revisar lógica.")

In [ ]:
# ==============================================================================
# Celda 9: Exportación de Datos Finales a Formato Parquet
# ==============================================================================
# Guardamos el resultado en la carpeta processed usando el DataManager
dm.save_processed_data(df_monthly)

print("✅ Proceso de transformación completado. El archivo está listo para el modelado.")